> **For dataset creators only.** This notebook converts raw two-photon calcium-imaging .mat files into the preprocessed  bundles distributed as part of the Mouse vs. AI Track 2 dataset. It requires private raw recording data not included in this repository. Reviewers and users of the benchmark do not need to run this notebook — the preprocessed data is available for download at [https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/YB8J31](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/YB8J31).

# Track 2 Preprocessing Pipeline

Documents the complete pipeline used to produce the preprocessed session bundles
released as part of the **Mouse vs AI Benchmark** (NeurIPS 2025).

**Input:** Raw session files in MATLAB HDF5 format (`.mat`, v7.3)  
**Output:** Neuron-filtered `.npz` bundles with provenance sidecars

```
mouseai-master/
├── [raw_sessions/]                          <- NOT included in release
│   └── {mouse}_{session}_mousevAI_perturbs.mat
├── [intermediate_npz/]                      <- intermediate, NOT released
│   └── {session}_preprocessed.npz           <- unfiltered, all neurons
└── mouse_vs_ai_data_preprocessed/           <- RELEASED DATA
    └── {session}_preprocessed/
        ├── {session}_preprocessed.npz        <- filtered (C0.005 neurons)
        ├── neuron_filter_mask.npy
        ├── original_neuron_indices.npy
        └── session_filter_metadata.json
```

### Pipeline stages

| Stage | Description |
|-------|-------------|
| 1 | Load `.mat`, process frames (resize, normalise, orientation fix), save `.npz` |
| 2 | Ridge regression (pixels → spikes), compute per-neuron R_visual |
| 3 | Apply C0.005 filter (R_visual ≥ 0.005 AND FR ≥ 100), save filtered bundles |

> **Note:** This notebook is for reproducibility documentation only.
> `mouse_vs_ai_data_preprocessed/` already contains the pipeline output.
> Raw `.mat` files are not included in the release.


## Setup

Edit the **User Settings** block below to match your local directory layout.


In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import json
import pickle
from datetime import datetime
from pathlib import Path

import cv2
import h5py
import numpy as np
from skimage.transform import resize as skimage_resize
from sklearn.linear_model import Ridge

# ── User Settings ─────────────────────────────────────────────────────────────
# Directory containing raw .mat session files (HDF5 v7.3, not in release)
# Set MAT_DIR to the folder containing raw .mat session files (not distributed).
MAT_DIR = Path("../../analyisis_old/JoeDATA/additional 6 sessions")

# Intermediate output: unfiltered .npz bundles (one per session)
NPZ_DIR = Path("../../reconstructed_npz")

# Final output: filtered session folders (= mouse_vs_ai_data_preprocessed/)
OUT_DIR = Path("../../mouse_vs_ai_data_preprocessed")

# Apply orientation fix? True for all 9 released sessions.
FIX_ORIENTATION = True

# ── Neuron filter thresholds ──────────────────────────────────────────────────
R_VISUAL_THRESHOLD = 0.005   # C0.005 criterion: min image→spike Pearson r
FR_THRESHOLD       = 100.0   # min total spike count on valid frames
FILTER_NAME        = "C0.005"

# ── Frame constants ───────────────────────────────────────────────────────────
TARGET_H, TARGET_W = 86, 155   # confirmed by neurons_s22.ipynb and .npz inspection
MAT_GROUP = "mousevAI_preprocessed_out"   # HDF5 group name inside .mat

ROOT = Path().resolve()
print(f"Notebook root : {ROOT}")
print(f"MAT_DIR       : {MAT_DIR.resolve()}")
print(f"NPZ_DIR       : {NPZ_DIR.resolve()}")
print(f"OUT_DIR       : {OUT_DIR.resolve()}")


---
## Stage 1 — Load raw `.mat` files and create `.npz` bundles

Each raw session is a MATLAB HDF5 v7.3 file containing the group
`mousevAI_preprocessed_out` with:

| Variable | MATLAB shape | h5py shape | dtype | Description |
|----------|-------------|------------|-------|-------------|
| `video` | `(T, H, W)` | `(W, H, T)` | uint8 | Stimulus video |
| `spikerates` | `(N, T)` | `(N, T)` | float64 | Binned spike counts, 0.1 s bins |
| `time` | `(1, T)` | `(1, T)` | float64 | Timestamps in seconds |
| `cellarea` | `(1, N)` | `(1, N)` | float64 | Brain-area index (1–9) |
| `blackouts` | `(1, T)` | `(1, T)` | uint8 | 0 = valid, 1 = excluded |

**Frame processing steps:**
1. Transpose `(W, H, T)` → `(T, H, W)` (h5py reverses MATLAB axis order)
2. RGB → grayscale (luminance weights, OpenCV convention)
3. Resize each frame to `86 × 155` px via bilinear interpolation
4. Add channel dim → `(T, 1, 86, 155)` ; normalise / 255 → float16

**Orientation fix** (`FIX_ORIENTATION = True`):  
The original preprocessing contained an H/W transposition. The fix (from
`Track2_avoid_kernel_death2.ipynb` Cell 7) uses `skimage.resize` to
re-interpolate to `(T, C, 155, 86)` then transposes axes `(0, 1, 3, 2)`
back to `(T, C, 86, 155)` with correct orientation.
All 9 released sessions were processed with this fix enabled.


In [ ]:
# ── Stage 1 helpers: .mat → .npz ────────────────────────────────────────────

def load_mat_bundle(mat_path):
    """Load one raw session .mat (HDF5 v7.3) into a dict of NumPy arrays."""
    with h5py.File(mat_path, 'r') as f:
        grp = f[MAT_GROUP]
        return {
            "video":      grp["video"][:],
            "spikerates": grp["spikerates"][:],
            "time":       grp["time"][:],
            "cellarea":   grp["cellarea"][:],
            "blackouts":  grp["blackouts"][:],
            "binwidth":   float(grp["binwidth"][0, 0]),
        }


def process_frames(video):
    """Convert h5py video (W, H, T) -> (T, 1, 86, 155) float16 [0, 1].

    1. Transpose (W, H, T) -> (T, H, W)
    2. RGB -> grayscale (luminance weights, OpenCV convention)
    3. Resize each frame to (TARGET_H, TARGET_W) via bilinear interpolation
    4. Add channel dim -> (T, 1, H, W); normalise /255 -> float16
    """
    video_t = video.transpose(2, 1, 0)          # (T, H, W)
    T = video_t.shape[0]
    frames = np.empty((T, TARGET_H, TARGET_W), dtype=np.float32)
    for i in range(T):
        frame = video_t[i]
        if frame.ndim == 3 and frame.shape[2] == 3:
            frame = cv2.cvtColor(frame.astype(np.uint8), cv2.COLOR_RGB2GRAY)
        elif frame.ndim == 3:
            frame = frame[:, :, 0]
        frames[i] = cv2.resize(frame.astype(np.uint8),
                               (TARGET_W, TARGET_H),
                               interpolation=cv2.INTER_LINEAR)
    frames = frames[:, np.newaxis, :, :]
    return (frames / 255.0).astype(np.float16)


def apply_orientation_fix(frames):
    """Correct the H/W transposition from the original pipeline.

    Source: Track2_avoid_kernel_death2.ipynb Cell 7.
    Interprets (T, C, 86, 155) as mislabelled (T, C, W, H), resizes to
    (T, C, H=155, W=86), then transposes axes back to (T, C, 86, 155).
    """
    N, C, W, H = frames.shape
    resized = skimage_resize(frames.astype(np.float32), (N, C, H, W),
                             anti_aliasing=True, preserve_range=True)
    return np.transpose(resized, (0, 1, 3, 2)).astype(frames.dtype)


def preprocess_session(mat_path, npz_path, fix_orient=True):
    """Preprocess one .mat -> .npz.  Returns a stats dict."""
    print(f'  {Path(mat_path).name}')
    raw    = load_mat_bundle(mat_path)
    T_full = raw['video'].shape[2]
    N      = raw['spikerates'].shape[0]

    frames = process_frames(raw['video'])
    if fix_orient:
        frames = apply_orientation_fix(frames)

    spikes    = raw['spikerates'].astype(np.float32)        # (N, T) float32
    time_arr  = raw['time'].squeeze().astype(np.float32)    # (T,)   float32
    cellarea  = raw['cellarea'].astype(np.uint8)            # (1, N) uint8
    blackouts = raw['blackouts'].astype(np.uint8)           # (1, T) uint8

    Path(npz_path).parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(str(npz_path),
                        frames=frames, spikes=spikes, time=time_arr,
                        cellarea=cellarea, blackouts=blackouts)

    n_bo = int(blackouts.sum())
    print(f'    T={T_full}  N={N}  blackouts={n_bo} ({100*n_bo/T_full:.1f}%)  '
          f'frames {frames.shape}  dtype={frames.dtype}')
    return {'stem': Path(mat_path).stem, 'T': T_full, 'N': N}


In [ ]:
# ── Run Stage 1 ─────────────────────────────────────────────────────────────
NPZ_DIR.mkdir(parents=True, exist_ok=True)

mat_files = sorted(MAT_DIR.glob('*.mat'))
print(f'Found {len(mat_files)} .mat file(s) in {MAT_DIR}')
print(f'fix_orientation = {FIX_ORIENTATION}\n')

step1_stats = []
for mat_path in mat_files:
    stem     = mat_path.stem.replace('_mousevAI_perturbs',
                                     '_mousevAI_perturbs_preprocessed')
    npz_path = NPZ_DIR / f'{stem}.npz'
    stats    = preprocess_session(mat_path, npz_path, fix_orient=FIX_ORIENTATION)
    step1_stats.append(stats)

print(f'\nStage 1 complete: {len(step1_stats)} session(s) saved to {NPZ_DIR}/')


---
## Stage 2 — Compute neuron filter (image → spike regression)

To select visually responsive neurons we fit a **linear Ridge regression**
model that predicts each neuron's spike train from the raw stimulus pixels.
The prediction quality — measured as per-neuron Pearson r — is called **R_visual**
and quantifies how much of a neuron's activity is linearly driven by the visual input.

**Algorithm (per session):**
1. Apply blackout mask → keep only valid frames
2. Flatten each frame `(1, 86, 155)` → pixel vector of length 13 330
3. Fit `Ridge(alpha=1.0)` mapping pixel matrix `X [T, 13 330]` → spike matrix `Y [T, N]`
   using **5-fold contiguous cross-validation** (time-series safe — no shuffling)
4. On each held-out fold: per-neuron Pearson r between predicted and observed spikes
5. Average r across the 5 folds → **R_visual [N]** (float64, no smoothing)
6. Total spike count on valid frames → **FR [N]** (float32)

Results are saved to a `.pkl` file used in Stage 3.  
⚠️ **Hyperparameters must not be changed** — they reproduce the official leaderboard.

> **Runtime:** ~10–30 minutes per session on CPU (13 330-dim regression × 5 folds).


In [ ]:
# ── Stage 2 helpers: image -> spike regression ───────────────────────────────
# DO NOT CHANGE — these reproduce the official Track 2 leaderboard scores
RIDGE_ALPHA = 1.0
N_CV_FOLDS  = 5


def contiguous_folds(T, n_folds):
    """Split T time steps into n_folds contiguous (train_idx, val_idx) pairs.

    Contiguous (non-shuffled) folds preserve the time-series structure of
    neural and visual data — random splits would allow temporal leakage.
    """
    block = T // n_folds
    splits = []
    for f in range(n_folds):
        start = f * block
        end   = (f + 1) * block if f < n_folds - 1 else T
        val   = np.arange(start, end)
        train = (np.concatenate([np.arange(0, start), np.arange(end, T)])
                 if start > 0 else np.arange(end, T))
        splits.append((train, val))
    return splits


def pearson_r(pred, true):
    """Per-neuron Pearson r between two [T, N] arrays.  Returns [N] float64."""
    pred = pred - pred.mean(axis=0, keepdims=True)
    true = true - true.mean(axis=0, keepdims=True)
    num  = np.sum(pred * true, axis=0)
    den  = np.sqrt(np.sum(pred ** 2, axis=0) * np.sum(true ** 2, axis=0)) + 1e-9
    r    = num / den
    r[np.isnan(r)] = 0
    return r


def compute_r_visual(frames, spikes, alpha=RIDGE_ALPHA, n_folds=N_CV_FOLDS):
    """Per-neuron visual responsiveness via 5-fold ridge regression.

    Parameters
    ----------
    frames : (T_valid, 1, H, W) float32  — valid frames (blackouts removed)
    spikes : (N, T_valid) float32         — spike counts on same valid frames

    Returns
    -------
    R_visual : (N,) float64
        Per-neuron mean Pearson r (image->spike), averaged over CV folds.
        float64 is preserved to match the original filter_raw.pkl dtype.
    """
    T = frames.shape[0]
    N = spikes.shape[0]
    X = frames.reshape(T, -1).astype(np.float32)   # (T, H*W = 13 330)

    folds   = contiguous_folds(T, n_folds)
    R_folds = np.zeros((n_folds, N), dtype=np.float64)

    for fi, (tr, va) in enumerate(folds):
        model = Ridge(alpha=alpha)
        model.fit(X[tr], spikes[:, tr].T)            # (T_tr, H*W) -> (T_tr, N)
        Yhat        = model.predict(X[va])            # (T_va, N)
        R_folds[fi] = pearson_r(Yhat, spikes[:, va].T)

    return np.mean(R_folds, axis=0)   # (N,) float64


def compute_filter_session(npz_path, verbose=True):
    """Load one .npz, apply blackout mask, run regression. Returns filter dict."""
    d      = np.load(npz_path, allow_pickle=True)
    mask   = d['blackouts'] == 0
    frames = d['frames'].astype('float32')[mask[0]]     # (T_valid, 1, H, W)
    spikes = d['spikes'].astype('float32')[:, mask[0]]  # (N, T_valid)
    T, N   = frames.shape[0], spikes.shape[0]

    if verbose:
        print(f'  {Path(npz_path).name}  T_valid={T}  N={N}', flush=True)

    R_visual = compute_r_visual(frames, spikes)
    FR       = np.sum(spikes, axis=1).astype(np.float32)

    if verbose:
        n_pass = int((R_visual >= R_VISUAL_THRESHOLD).sum())
        print(f'    R_visual: mean={R_visual.mean():.4f}  '
              f'n(R>={R_VISUAL_THRESHOLD})={n_pass}/{N} '
              f'({100*n_pass/N:.1f}%)', flush=True)

    return {'R_visual': R_visual, 'FR': FR, 'n_neurons': N, 'mode': 'raw'}


In [ ]:
# ── Run Stage 2 ─────────────────────────────────────────────────────────────
print(f'Computing image->spike filter')
print(f'  Ridge alpha={RIDGE_ALPHA},  CV folds={N_CV_FOLDS}')
print(f'  (~10-30 min per session on CPU)\n')

filter_data = {}
for npz_path in sorted(NPZ_DIR.glob('*.npz')):
    filter_data[npz_path.stem] = compute_filter_session(npz_path)

# Save intermediate filter dict
filter_pkl = NPZ_DIR / 'filter_raw.pkl'
with open(filter_pkl, 'wb') as fh:
    pickle.dump(filter_data, fh)
print(f'\nStage 2 complete: filter saved to {filter_pkl}')
print(f'  Sessions: {sorted(filter_data.keys())}')


---
## Stage 3 — Apply neuron filter and save final bundles

Using the per-neuron statistics from Stage 2, we apply the **C0.005 filter**:

> Keep neuron *n* if and only if **R_visual ≥ 0.005** AND **FR ≥ 100**

where FR is the **total spike count on valid (non-blackout) frames**.  
This selects neurons that are both sufficiently active and have activity
that is linearly predictable from the visual stimulus.

**Per-session outputs** (written to `OUT_DIR/{session}/`):

| File | Shape / type | Description |
|------|-------------|-------------|
| `{session}.npz` | — | Filtered bundle: `frames`, `spikes (N_kept × T)`, `time`, `cellarea`, `blackouts` |
| `neuron_filter_mask.npy` | `bool[N_orig]` | True = neuron passes filter |
| `original_neuron_indices.npy` | `uint32[N_kept]` | Indices in original population |
| `session_filter_metadata.json` | JSON | Provenance: thresholds, counts, timestamps |

`frames`, `time`, and `blackouts` are copied **unchanged**;  
only `spikes` (axis 0) and `cellarea` (axis 1) are reduced.


In [ ]:
# ── Stage 3 helpers: apply filter -> final bundles ──────────────────────────

def apply_filter_session(npz_path, filter_entry, out_dir,
                         r_thr=R_VISUAL_THRESHOLD,
                         fr_thr=FR_THRESHOLD,
                         filter_name=FILTER_NAME):
    """Apply neuron filter to one session and write outputs.

    Writes to out_dir/{stem}/:
      {stem}.npz                  -- filtered bundle
      neuron_filter_mask.npy      -- bool[N_orig], True = kept
      original_neuron_indices.npy -- uint32[N_kept], indices in original population
      session_filter_metadata.json

    Returns the provenance metadata dict.
    """
    stem = Path(npz_path).stem
    d    = np.load(npz_path, allow_pickle=True)

    spikes    = d['spikes']            # (N_orig, T_full)
    frames    = d['frames']            # (T_full, 1, H, W)  -- unchanged
    blackouts = d['blackouts']         # (1, T_full)         -- unchanged
    time_arr  = d['time']     if 'time'     in d else None
    cellarea  = d['cellarea'] if 'cellarea' in d else None

    N_orig, T_full = spikes.shape

    # Build neuron mask
    R_visual = np.asarray(filter_entry['R_visual'])
    FR       = np.asarray(filter_entry['FR'])
    mask     = (R_visual >= r_thr) & (FR >= fr_thr)   # bool[N_orig]
    indices  = np.where(mask)[0].astype(np.uint32)     # uint32[N_kept]
    N_kept   = int(mask.sum())

    print(f'  {stem}')
    print(f'    N: {N_orig} -> {N_kept}  ({100*N_kept/N_orig:.1f}% pass {filter_name})')

    # Filter neural arrays
    spikes_filt   = spikes[mask, :]                        # (N_kept, T_full)
    cellarea_filt = cellarea[:, mask] if cellarea is not None else None  # (1, N_kept)

    # Provenance metadata
    meta = {
        'session_id':                stem,
        'filter_name':               filter_name,
        'r_visual_threshold':        float(r_thr),
        'fr_threshold':              float(fr_thr),
        'original_num_neurons':      N_orig,
        'kept_num_neurons':          N_kept,
        'excluded_num_neurons':      N_orig - N_kept,
        'fraction_kept':             round(N_kept / N_orig, 6),
        'num_timepoints_full':       T_full,
        'visual_frame_shape':        list(frames.shape[1:]),
        'neural_array_shape_before': list(spikes.shape),
        'neural_array_shape_after':  list(spikes_filt.shape),
        'date_created':              datetime.utcnow().isoformat() + 'Z',
        'notes': 'C0.005 filter: FR>=100 total spikes AND R_visual>=0.005',
    }

    # Write outputs
    session_dir = Path(out_dir) / stem
    session_dir.mkdir(parents=True, exist_ok=True)

    save_kw = dict(frames=frames, spikes=spikes_filt, blackouts=blackouts)
    if time_arr     is not None: save_kw['time']     = time_arr
    if cellarea_filt is not None: save_kw['cellarea'] = cellarea_filt

    np.savez_compressed(str(session_dir / f'{stem}.npz'), **save_kw)
    np.save(str(session_dir / 'neuron_filter_mask.npy'),      mask)
    np.save(str(session_dir / 'original_neuron_indices.npy'), indices)
    with open(session_dir / 'session_filter_metadata.json', 'w') as fh:
        json.dump(meta, fh, indent=2)

    print(f'    -> {stem}/ (npz + 3 sidecar files)')
    return meta


In [ ]:
# ── Run Stage 3 ─────────────────────────────────────────────────────────────
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Applying {FILTER_NAME} filter '
      f'(R_visual>={R_VISUAL_THRESHOLD} AND FR>={FR_THRESHOLD:.0f})\n')

all_meta = []
for npz_path in sorted(NPZ_DIR.glob('*.npz')):
    stem = npz_path.stem
    if stem not in filter_data:
        print(f'  WARNING: no filter entry for {stem} -- skipping.')
        continue
    meta = apply_filter_session(
        npz_path, filter_data[stem], OUT_DIR,
        r_thr=R_VISUAL_THRESHOLD, fr_thr=FR_THRESHOLD,
    )
    all_meta.append(meta)

print(f'\nStage 3 complete: {len(all_meta)} session(s) saved to {OUT_DIR}/')


---
## Summary


In [ ]:
# ── Dataset summary ─────────────────────────────────────────────────────────
print('=' * 68)
print('  Preprocessing complete')
print('=' * 68)

total_kept = total_orig = total_T = 0
rows = []
for meta in all_meta:
    total_kept += meta['kept_num_neurons']
    total_orig += meta['original_num_neurons']
    total_T    += meta['num_timepoints_full']
    rows.append((
        meta['session_id'],
        meta['num_timepoints_full'],
        meta['original_num_neurons'],
        meta['kept_num_neurons'],
        f"{100*meta['fraction_kept']:.1f}%",
    ))

hdr = f'  {"Session":<55} {"T_full":>7} {"N_orig":>7} {"N_kept":>7} {"kept%":>6}'
sep = f'  {"-"*55} {"-"*7} {"-"*7} {"-"*7} {"-"*6}'
print(hdr)
print(sep)
for r in rows:
    print(f'  {r[0]:<55} {r[1]:>7} {r[2]:>7} {r[3]:>7} {r[4]:>6}')
print(sep)
print(f'  {"TOTAL":<55} {total_T:>7} {total_orig:>7} {total_kept:>7}')
print()
print(f'  Filter  : R_visual >= {R_VISUAL_THRESHOLD}  AND  FR >= {FR_THRESHOLD:.0f}  ({FILTER_NAME})')
print(f'  Sessions: {len(all_meta)}')
print(f'  Total T : {total_T:,} bins  ({total_T/600:.1f} min at 0.1 s/bin)')
print(f'  N_kept  : {total_kept:,} neurons  (from {total_orig:,} original)')
